# World Bank Brazil adaptation-screen release v3

This restartable notebook wraps the release build, documents the evidence boundary, and validates the cleaned project and action-crosswalk outputs. It uses retained official API responses by default; set `REFRESH = True` to retrieve the current official responses. The selection is a research screen, not an attribution of project commitments to adaptation finance.

In [1]:
from pathlib import Path
import importlib.util
import pandas as pd

HERE = Path.cwd()
if not (HERE / 'build_brazil_adaptation_release.py').exists():
    raise RuntimeError('Run this notebook from releases/v3')
spec = importlib.util.spec_from_file_location('release_builder', HERE / 'build_brazil_adaptation_release.py')
builder = importlib.util.module_from_spec(spec)
spec.loader.exec_module(builder)
REFRESH = False

## Extract, screen, standardise and map

The builder obtains the Brazil country response and seven query-result evidence sets when refreshing. It preserves project lifecycle status and dates separately, translates only the one Portuguese source title, leaves the adaptation-finance amount blank, and creates an explicit mapping or none row for every released project.

In [2]:
projects, mappings = builder.build(refresh=REFRESH)
projects.head(3)

,project_id,project_name,project_name_pt,project_name_translation_status,country_code,country_name,region_name,lifecycle_status,has_board_approval_evidence,board_approval_date,...,ida_commitment_amount_usd,grant_amount_usd,environmental_assessment_category_code,project_url,official_api_query_matches,adaptation_screen_evidence,adaptation_screen_confidence,adaptation_finance_amount_usd,adaptation_finance_amount_status,source_retrieved_date
0,P006351,Northeast Rural Development Project (03) - BAHIA,,source_english,BR,Brazil,Latin America and Caribbean,Closed,True,1986-10-21,...,0,<NA>,C,https://projects.worldbank.org/en/projects-ope...,disaster_risk | drought,multiple_official_api_query_matches,medium,<NA>,not_reported_do_not_infer_from_project_commitment,2026-09-25
1,P006417,Northeast Urban Flood Reconstruction Project,,source_english,BR,Brazil,Latin America and Caribbean,Closed,True,1985-12-19,...,0,<NA>,,https://projects.worldbank.org/en/projects-ope...,flood,explicit_project_title,high,<NA>,not_reported_do_not_infer_from_project_commitment,2026-09-25
2,P006475,Land Reform and Poverty Alleviation Pilot Project,,source_english,BR,Brazil,Latin America and Caribbean,Closed,True,1997-04-22,...,0,<NA>,B,https://projects.worldbank.org/en/projects-ope...,drought | flood,multiple_official_api_query_matches,medium,<NA>,not_reported_do_not_infer_from_project_commitment,2026-09-25


## Validation

Approval is not implementation, closing is not disbursement, and the published whole-project commitment is not an adaptation amount. These assertions enforce that boundary as well as the released row counts.

In [3]:
assert len(projects) == projects['project_id'].nunique() == 55
assert projects['lifecycle_status'].value_counts().to_dict() == {'Closed': 26, 'Active': 22, 'Pipeline': 5, 'Dropped': 2}
assert int(projects['total_commitment_amount_usd'].fillna(0).sum()) == 8_679_030_000
assert projects['adaptation_finance_amount_usd'].isna().all()
assert (~projects['has_disbursement_data']).all()
assert len(mappings) == mappings['project_id'].nunique() == 55
assert set(mappings['project_id']) == set(projects['project_id'])
assert mappings['action_id'].ne('').sum() == 9
assert mappings['allocated_amount_usd'].isna().all()
assert set(mappings.loc[mappings['action_id'].ne(''), 'confidence']) == {'medium'}
print({'projects': len(projects), 'mapped': int(mappings['action_id'].ne('').sum()), 'commitment_validation_total_usd': int(projects['total_commitment_amount_usd'].fillna(0).sum())})

{'projects': 55, 'mapped': 9, 'commitment_validation_total_usd': 8679030000}
